# DAC Box controller (simple)

Read and ramp the flux DAC currents on the DAC Box units over the network, in mA.
The units are numbered in order: the first unit is **DAC 1-4**, the second **DAC 5-8**.

Self-contained: runs on the base environment, with the driver in the collapsed cells below.
It does not import malab. The older malab in the base install ramps these boxes about
1000x too fast (it gives the step delay in ms where the firmware expects us).

**Before you start**
- A box takes **one connection at a time**.
- Currents are worked out from the codes the box tracks and each DAC's load resistor
  (I = V / 1kOhm). 

**Safety rules**
- Every move is a ramp in the box's firmware, at most 0.1 mA/s.
- Max is 10mA with resistor and is enforced by code
- A DAC is initialised only when it is cold (never used, held at 0) or reads zero.
- **To stop a ramp, interrupt the kernel** (the stop button). The box aborts the ramp and the
  output **holds where it is**. Nothing moves it afterwards.
- The cells that move a DAC do nothing until you set a target, so *Run All* never ramps anything.

## Settings

In [2]:
# Units in DAC order: the first is DAC 1-4, the second DAC 5-8.
UNITS = [("unit1", "xxx.xxx.xxx.xxx"),     # DAC 1-4
         ("unit2", "xxx.xxx.xxx.xxx")]     # DAC 5-8

# Load resistor of each DAC in ohm (I = 1000 V / R, in mA). None: no mA for that DAC.
RESISTORS_OHM = [1000, 1000, 1000, 1000,      # DAC 1-4
                 1000, 1000, 1000, None]      # DAC 5-8 (DAC 8 unused)

LABELS = ["Q1 - On-Chip", "Q2 - On-Chip", "Q3 - On-Chip", "Q4 - On-Chip",
          "Q5 - On-Chip", "Q6 - On-Chip", "Q7 - On-Chip", "Unused"]

MAX_RATE_MA_S = 0.1     # ramp rate limit
WARN_MA = 10.0          # |I| above this warns
LIMIT_MA = 10.0         # |I| above this is refused

## Driver

Collapsed: click the cells to expand them. Run them once after starting the kernel.

In [3]:
# DAC Box driver, part 1: one unit (four AD5780 channels) over its network connection.
# Ported from the DAC Box fork of malab (instruments/ad5780.py, _line_socket.py) so this
# notebook needs nothing beyond the base environment. Do NOT use the older malab in the
# base install for these boxes: its AD5780.ramp gives the step delay in ms where this
# firmware expects us, so it ramps about 1000x too fast.
import math
import socket
import time

V_PER_BIT = 20.0 / 2**18                    # nominal: 18 bits over +-10 V
MAX_CODE = 2**18 - 1
SOURCES = ("unknown", "readback", "write")  # VALUES source; unknown = cold (never initialised, clamped at 0 V)


class DacBoxError(RuntimeError):
    """A DAC Box command failed or was refused. Nothing is retried: the output stays where it is."""


class DacBox:
    """
    One DAC Box unit: four channels behind one network connection (telnet port 23).

    The firmware allows one client at a time, and aborts a running ramp when that
    client disconnects - the output then holds where it is. READ is refused while a
    ramp runs, so outputs are read with VALUES (the codes the box tracks).
    """
    default_port = "xxxx"

    def __init__(self, name, address, timeout=3.0):
        host, _, port = address.partition(":")
        self.name, self.address = name, address
        self.host, self.port = host, int(port) if port else self.default_port
        self.timeout = timeout
        self.sock = None
        self._rx = b""
        self._cal = {}

    def __repr__(self):
        return f"DacBox({self.name!r}, {self.address!r}, {'connected' if self.connected else 'not connected'})"

    # ---------------------------------------------------------------- connection
    @property
    def connected(self):
        return self.sock is not None

    def connect(self):
        """Open the connection and check the box answers as an AD5780. Changes no output."""
        if self.sock is not None:
            return self
        try:
            self.sock = socket.create_connection((self.host, self.port), self.timeout)
        except OSError as e:
            raise DacBoxError(f"{self.name} ({self.address}): cannot connect: {e}") from e
        self.sock.settimeout(self.timeout)
        self._rx, self._cal = b"", {}
        first = self._readline()
        if first.startswith("ERR!"):
            self.close()
            raise DacBoxError(f"{self.name} ({self.address}): {first} Another program holds this box "
                              "(the old DAC dashboard, the qlab GUI or another notebook): close it, then connect again.")
        second = self._readline()
        if (first, second) != ("Welcome to the Telnet server!", "Type HELP for instructions."):
            self.close()
            raise DacBoxError(f"{self.name} ({self.address}): unexpected welcome {first!r} {second!r}")
        if self.query("ID") != "AD5780":
            self.close()
            raise DacBoxError(f"{self.name} ({self.address}) is not an AD5780 DAC Box")
        return self

    def close(self):
        """Close the connection. Outputs hold; a running ramp is aborted and holds where it is."""
        sock, self.sock, self._rx = self.sock, None, b""
        if sock is not None:
            sock.close()

    # ---------------------------------------------------------------- line protocol
    def _readline(self):
        while b"\n" not in self._rx:
            try:
                data = self.sock.recv(4096)
            except OSError as e:                # includes timeouts: the outcome is unknown, so close
                self.close()
                raise DacBoxError(f"{self.name}: no reply ({e}); connection closed") from e
            if not data:
                self.close()
                raise DacBoxError(f"{self.name}: the box closed the connection")
            self._rx += data
        line, _, self._rx = self._rx.partition(b"\n")
        return line.rstrip(b"\r").decode("ascii", "replace")

    def query(self, command):
        """Send one command line and return the reply line; an ERR! reply raises."""
        if self.sock is None:
            raise DacBoxError(f"{self.name} ({self.address}) is not connected: run dacs.connect()")
        try:
            self.sock.sendall(command.encode("ascii") + b"\n")
        except OSError as e:
            self.close()
            raise DacBoxError(f"{self.name}: send failed ({e}); connection closed") from e
        reply = self._readline()
        if reply.startswith("ERR!"):
            raise DacBoxError(f"{self.name}: {command!r} refused: {reply}")
        return reply

    # ---------------------------------------------------------------- calibration
    def calibration(self, ch):
        """(volts at code 0, volts per code) of channel ch (1-4), from the box's own calibration."""
        if ch not in self._cal:
            reply = self.query(f"CAL {ch}")
            try:
                label, c, offset, step, zero, measured = reply.split()
                c, offset, step, zero, measured = map(int, (c, offset, step, zero, measured))
                if (label != "CAL" or c != ch or not -12_000_000_000 <= offset <= -8_000_000_000
                        or not 60_000_000 <= step <= 100_000_000):
                    raise ValueError
            except ValueError:
                self.close()
                raise DacBoxError(f"{self.name}: invalid calibration {reply!r}; connection closed") from None
            nominal = (offset, step, zero, measured) == (-10_000_000_000, 76_293_945, 131_072, 0)
            self._cal[ch] = (offset * 1e-9, V_PER_BIT if nominal else step * 1e-12)
        return self._cal[ch]

    def code_to_volts(self, ch, code):
        v0, dv = self.calibration(ch)
        return v0 + code * dv

    def volts_to_code(self, ch, volts):
        if not -10 <= volts <= 10:
            raise DacBoxError(f"{volts} V is outside +-10 V")
        v0, dv = self.calibration(ch)
        raw = (volts - v0) / dv
        if raw < -1 or raw > MAX_CODE + 1:
            raise DacBoxError(f"{volts} V is outside channel {ch}'s range")
        return min(MAX_CODE, max(0, int(math.floor(raw + 0.5))))

    # ---------------------------------------------------------------- commands
    def values(self):
        """[(code or None, source)] of channels 1-4: the box's tracked codes (also during a ramp)."""
        reply = self.query("VALUES")
        p = reply.split()
        try:
            if len(p) != 9 or p[0] != "VALUES":
                raise ValueError
            out = []
            for i in range(4):
                code, source = int(p[2 * i + 1]), int(p[2 * i + 2])
                out.append((code if source else None, SOURCES[source]))
            return out
        except (ValueError, IndexError):
            self.close()
            raise DacBoxError(f"{self.name}: malformed VALUES {reply!r}; connection closed") from None

    def status(self):
        """(state, channel, target code) of the box's ramp; state IDLE/RUNNING/COMPLETE/ABORTED/FAILED."""
        reply = self.query("STATUS")
        p = reply.split()
        if len(p) != 4 or p[0] != "RAMP" or p[1] not in ("IDLE", "RUNNING", "COMPLETE", "ABORTED", "FAILED"):
            self.close()
            raise DacBoxError(f"{self.name}: malformed STATUS {reply!r}; connection closed")
        return p[1], int(p[2]), int(p[3])

    def initialize(self, ch):
        """INIT one channel: a cold channel is released at its calibrated zero; active outputs are kept."""
        reply = self.query(f"INIT {ch}")
        if not reply.startswith("Initialized"):
            self.close()
            raise DacBoxError(f"{self.name}: unexpected INIT reply {reply!r}; connection closed")
        return reply

    def start_ramp(self, ch, target_code, volts_per_s, step_codes=10):
        """Start the box's own ramp of channel ch to target_code at volts_per_s; returns at once.

        The firmware moves step_codes every delay_us MICROseconds (not ms)."""
        _, dv = self.calibration(ch)
        delay_us = max(1, math.ceil(step_codes * dv / volts_per_s * 1e6))
        if delay_us > 2_147_483_647:
            raise DacBoxError("ramp rate too slow for the firmware")
        command = f"RAMP {ch} {target_code} {step_codes} {delay_us}"
        reply = self.query(command)
        if reply != f"RAMP ACCEPTED {ch} {target_code} {step_codes} {delay_us}":
            self.close()                         # acceptance unknown: closing aborts any ramp it started
            raise DacBoxError(f"{self.name}: {command!r} got {reply!r}; connection closed")
        return delay_us

In [4]:
# DAC Box driver, part 2: all units in order, currents in mA, and the safety rules.
class FluxDACs:
    """
    The DAC Box units in order: the first unit is DAC 1-4, the second DAC 5-8, ...

    Currents are in mA through each channel's load resistor: I = 1000 V / R. They come
    from the codes the box tracks, not from a meter.

    Safety rules:
      * ramps run in the box's firmware at <= max_rate_mA_s; nothing jumps to a new value
      * |I| above warn_mA prints a warning; above limit_mA is refused (either sign)
      * a channel is initialised only when it is cold (clamped at 0) or reads zero
      * Interrupt (stop button) during a ramp closes that unit's connection: the box
        aborts the ramp and the output HOLDS where it is; it is never moved after
        a stop or an error
    """

    def __init__(self, units, resistors_ohm, labels=None, max_rate_mA_s=0.1, warn_mA=15.0,
                 limit_mA=20.0, zero_tol_mA=0.001, timeout=3.0):
        self.boxes = [DacBox(name, address, timeout) for name, address in units]
        n = 4 * len(self.boxes)
        if len(resistors_ohm) != n:
            raise ValueError(f"resistors_ohm needs {n} entries (one per DAC), got {len(resistors_ohm)}")
        self.resistors = list(resistors_ohm)
        self.labels = list(labels) if labels else [""] * n
        self.max_rate_mA_s, self.warn_mA, self.limit_mA = max_rate_mA_s, warn_mA, limit_mA
        self.zero_tol_mA = zero_tol_mA

    @property
    def n(self):
        return 4 * len(self.boxes)

    # ---------------------------------------------------------------- connection
    def connect(self):
        """Connect every unit (reads only, changes no output); prints each unit's result."""
        for i, box in enumerate(self.boxes):
            dacs = f"DAC {4 * i + 1}-{4 * i + 4}"
            try:
                box.connect()
                print(f"{box.name} ({box.address}) {dacs}: connected")
            except DacBoxError as e:
                print(f"{dacs}: NOT connected - {e}")
        return self

    def close(self):
        """Disconnect every unit. Outputs hold."""
        for box in self.boxes:
            box.close()
        print("Disconnected; outputs hold.")

    def locate(self, dac):
        """(unit, channel 1-4 on it) of global DAC number dac (1..n)."""
        if not 1 <= dac <= self.n:
            raise ValueError(f"DAC must be 1..{self.n}")
        return self.boxes[(dac - 1) // 4], (dac - 1) % 4 + 1

    # ---------------------------------------------------------------- reading
    def read(self, dac):
        """dict(code, source, volts, mA) of one DAC; code/volts/mA are None when cold."""
        box, ch = self.locate(dac)
        code, source = box.values()[ch - 1]
        volts = None if code is None else box.code_to_volts(ch, code)
        r = self.resistors[dac - 1]
        mA = None if volts is None or not r else 1000 * volts / r
        return {"dac": dac, "code": code, "source": source, "volts": volts, "mA": mA}

    def read_mA(self, dac):
        """Present current of one DAC in mA (None when cold or with no resistor)."""
        return self.read(dac)["mA"]

    def show(self):
        """Every DAC's present output, unit by unit, and each unit's ramp state."""
        for i, box in enumerate(self.boxes):
            if not box.connected:
                print(f"{box.name} ({box.address}) DAC {4 * i + 1}-{4 * i + 4}: not connected")
                continue
            state, ch, _ = box.status()
            ramp = f"ramping DAC {4 * i + ch}" if state == "RUNNING" else f"ramp {state.lower()}"
            print(f"{box.name} ({box.address})  {ramp}")
            for dac in range(4 * i + 1, 4 * i + 5):
                r = self.read(dac)
                label = self.labels[dac - 1]
                if r["code"] is None:
                    value = "cold (never initialised, at 0)"
                elif r["mA"] is None:
                    value = f"{r['volts']:+.6f} V   (no resistor: no mA)"
                else:
                    value = f"{r['mA']:+.4f} mA   ({r['volts']:+.6f} V, code {r['code']})"
                print(f"  DAC {dac:<2} {label:<14} {value}")

    # ---------------------------------------------------------------- moving
    def initialize(self, dac):
        """Initialise one DAC, only if it is cold or reads zero (it is released at 0)."""
        r = self.read(dac)
        if r["code"] is not None and (r["mA"] is None or abs(r["mA"]) > self.zero_tol_mA):
            raise DacBoxError(f"DAC {dac} reads {r['mA']} mA: initialise only at zero (ramp it to 0 first)")
        box, ch = self.locate(dac)
        print(f"DAC {dac}: {box.initialize(ch)}")

    def ramp_mA(self, dac, target_mA, rate_mA_s=0.05, settle_s=0.0):
        """
        Ramp one DAC to target_mA at rate_mA_s in the box's firmware, wait until it is
        there (progress printed), then settle_s more. Returns the reached current in mA.
        A cold DAC is first initialised at 0. Interrupt the kernel to stop: the output
        holds where it is.
        """
        r_ohm = self.resistors[dac - 1]
        if not r_ohm:
            raise DacBoxError(f"DAC {dac} has no load resistor set: no mA without one")
        if not 0 < rate_mA_s <= self.max_rate_mA_s:
            raise DacBoxError(f"rate {rate_mA_s} mA/s is outside (0, {self.max_rate_mA_s}] mA/s")
        if abs(target_mA) > self.limit_mA:
            raise DacBoxError(f"DAC {dac}: {target_mA} mA is beyond the {self.limit_mA} mA limit: refused")
        if abs(target_mA) > self.warn_mA:
            print(f"WARNING: DAC {dac} to {target_mA} mA is above {self.warn_mA} mA")
        box, ch = self.locate(dac)
        state, busy_ch, _ = box.status()
        if state == "RUNNING":
            raise DacBoxError(f"{box.name} is already ramping DAC {4 * self.boxes.index(box) + busy_ch}")

        now = self.read(dac)
        if now["code"] is None:
            print(f"DAC {dac} is cold: initialising at 0")
            self.initialize(dac)
            now = self.read(dac)
        start = now["mA"]
        target_code = box.volts_to_code(ch, target_mA * r_ohm / 1000)
        if now["code"] == target_code:
            print(f"DAC {dac} is already at {start:+.4f} mA")
            return start

        duration = abs(target_mA - start) / rate_mA_s
        print(f"DAC {dac}: {start:+.4f} -> {target_mA:+.4f} mA at {rate_mA_s} mA/s (~{duration:.0f} s)")
        box.start_ramp(ch, target_code, rate_mA_s * r_ohm / 1000)
        deadline = time.monotonic() + 1.5 * duration + 30
        try:
            while True:
                time.sleep(0.5)
                state, _, _ = box.status()
                present = self.read_mA(dac)
                print(f"\r  {present:+.4f} mA  {state.lower():<9}", end="")
                if state == "COMPLETE":
                    break
                if state != "RUNNING":
                    raise DacBoxError(f"DAC {dac}: ramp ended {state}; output held at {present:+.4f} mA")
                if time.monotonic() > deadline:
                    box.close()                  # aborts the ramp: the output holds
                    raise DacBoxError(f"DAC {dac}: ramp did not finish in time; stopped and held at "
                                      f"{present:+.4f} mA (unit disconnected: run dacs.connect())")
        except KeyboardInterrupt:
            box.close()                          # the box aborts the ramp; the output holds
            print()
            try:
                box.connect()
                held = f"{self.read_mA(dac):+.4f} mA"
            except DacBoxError as e:
                held = f"unknown ({e})"
            print(f"STOPPED: DAC {dac} holds at {held}. It will not be moved.")
            raise
        print()
        end = self.read(dac)
        if end["code"] != target_code:
            raise DacBoxError(f"DAC {dac}: ramp reported complete at code {end['code']}, asked {target_code}")
        if settle_s:
            time.sleep(settle_s)
        print(f"DAC {dac} at {end['mA']:+.4f} mA")
        return end["mA"]

    def ramp_all_to_zero(self, rate_mA_s=0.05):
        """Ramp every DAC that is not already at zero back to 0 mA, one after another."""
        for dac in range(1, self.n + 1):
            box, _ = self.locate(dac)
            if not box.connected or not self.resistors[dac - 1]:
                continue
            mA = self.read_mA(dac)
            if mA is not None and abs(mA) > self.zero_tol_mA:
                self.ramp_mA(dac, 0.0, rate_mA_s)
        print("All DACs at 0 mA (or cold).")

## Connect

Connects every unit. This only reads; it changes no output.

In [10]:
dacs = FluxDACs(UNITS, RESISTORS_OHM, LABELS,
                max_rate_mA_s=MAX_RATE_MA_S, warn_mA=WARN_MA, limit_mA=LIMIT_MA).connect()

unit1 (xxx.xxx.xxx.xxx) DAC 1-4: connected
unit2 (xxx.xxx.xxx.xxx) DAC 5-8: connected


## Read

In [11]:
dacs.show()

unit1 (xxx.xxx.xxx.xxx)  ramp complete
  DAC 1  Q1 - On-Chip   +5.0000 mA   (+5.000000 V, code 196608)
  DAC 2  Q2 - On-Chip   +0.0000 mA   (+0.000000 V, code 131072)
  DAC 3  Q3 - On-Chip   +0.0000 mA   (+0.000000 V, code 131072)
  DAC 4  Q4 - On-Chip   +0.0000 mA   (+0.000000 V, code 131072)
unit2 (xxx.xxx.xxx.xxx)  ramp complete
  DAC 5  Q5 - On-Chip   +0.0000 mA   (+0.000000 V, code 131072)
  DAC 6  Q6 - On-Chip   +0.0000 mA   (+0.000000 V, code 131072)
  DAC 7  Q7 - On-Chip   +0.0000 mA   (+0.000000 V, code 131072)
  DAC 8  Unused         +0.000000 V   (no resistor: no mA)


In [7]:
dacs.read_mA(1)     # one DAC, in mA (None if it is cold)

0.0

## Ramp

Set `TARGET_MA` and run. A cold DAC is first initialised at 0. Interrupt the kernel to stop:
the output holds where it is.

In [12]:
DAC = 1             # 1-4: first unit, 5-8: second unit
TARGET_MA = 0    # e.g. 0.5; None does nothing
RATE_MA_S = 0.05    # at most MAX_RATE_MA_S

if TARGET_MA is not None:
    dacs.ramp_mA(DAC, TARGET_MA, rate_mA_s=RATE_MA_S)

DAC 1: +5.0000 -> +0.0000 mA at 0.05 mA/s (~100 s)
  +0.0000 mA  complete 
DAC 1 at +0.0000 mA


### Several DACs, one after another

In [ ]:
TARGETS = {}        # DAC: mA, e.g. {1: 0.5, 5: -0.2}; empty does nothing

for dac, mA in TARGETS.items():
    dacs.ramp_mA(dac, mA, rate_mA_s=RATE_MA_S)
if TARGETS:
    dacs.show()

### Initialise a cold DAC

Only needed on its own if you want a cold DAC released at 0 without ramping it. A ramp does
this itself. Refused unless the DAC is cold or reads zero.

In [ ]:
# dacs.initialize(1)

## Back to zero

In [11]:
dacs.ramp_all_to_zero(rate_mA_s=0.05)     # every DAC not at 0, one after another

DAC 1: +1.0000 -> +0.0000 mA at 0.05 mA/s (~20 s)
  +0.0000 mA  complete 
DAC 1 at +0.0000 mA
All DACs at 0 mA (or cold).


## Disconnect

Frees the boxes for another program. Outputs hold.

In [12]:
dacs.close()

Disconnected; outputs hold.
